In [11]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report

# =================================================================
# PASO 1: Cargar los Datos Ya Preprocesados
# =================================================================
df = pd.read_csv("data/partidos_entrenamiento.csv", parse_dates=["date"])

# =================================================================
# PASO 2: Definir las Características (X) y el Objetivo (y)
# =================================================================
# Incluimos de forma estricta los nuevos predictores que calcula tu script
features = [
    "goles_totales_marcados_local_10",
    "goles_totales_concedidos_local_10",
    "forma_puntos_local_10",
    "goles_totales_marcados_visitante_10",
    "goles_totales_concedidos_visitante_10",
    "forma_puntos_visitante_10",
    "dif_goles_marcados_10",
    "dif_goles_concedidos_10",
    "dif_forma_puntos_10",
    "puntos_fifa_local",
    "puntos_fifa_visitante",
    "dif_puntos_fifa",
    "ranking_pos_local",
    "ranking_pos_visitante",
    "importancia_torneo",
]

df = df[df["date"] < "2026-05-01"]

df_train = df[(df["date"] < "2023-06-01") & (df["neutral"] == True)].copy()
X_train = df_train[features]
y_train = df_train["resultado"]
df_test = df[(df["date"] >= "2023-06-01") & (df["neutral"] == True)].copy()
X_test = df_test[features]
y_test = df_test["resultado"]
y_test


modelo_gb = HistGradientBoostingClassifier(
    max_iter=200,  # reducimos arboles
    learning_rate=0.05,  # pasos más cortos
    class_weight="balanced",
    max_depth=4,  # limitamos la profundidad
    min_samples_leaf=50,  # Exigimos que al menos 30 partidos cumplan una regla para darla por válida
    random_state=42,
)

# =================================================================
# PASO 2: Entrenar el modelo (Aprender de los datos de entrenamiento)
# =================================================================
# Ajustamos el modelo usando las pistas (X_train) y las respuestas reales (y_train)
modelo_gb.fit(X_train, y_train)

# =================================================================
# PASO 3: Realizar las predicciones
# =================================================================
# Le pedimos al modelo que intente adivinar los resultados del 20% de partidos reservados
predicciones = modelo_gb.predict(X_test)

# =================================================================
# PASO 4: Evaluar los resultados en consola
# =================================================================
# Calculamos el porcentaje total de aciertos
precision = accuracy_score(y_test, predicciones)

print(f"La precisión es: {precision * 100:.2f}%")
print("\nPor cada resultado (1=Local, 0=Empate, 2=Visitante):")
print(classification_report(y_test, predicciones))


La precisión es: 53.41%

Por cada resultado (1=Local, 0=Empate, 2=Visitante):
              precision    recall  f1-score   support

         0.0       0.32      0.31      0.32       281
         1.0       0.65      0.54      0.59       438
         2.0       0.57      0.69      0.62       380

    accuracy                           0.53      1099
   macro avg       0.51      0.51      0.51      1099
weighted avg       0.54      0.53      0.53      1099



In [12]:
import pandas as pd
import numpy as np

# 1. Cargamos el archivo limpio que generó tu script de preprocesamiento
df_entrenamiento = pd.read_csv('data/partidos_entrenamiento.csv', parse_dates=['date'])

# 2. Creamos un diccionario vacío para guardar el estado de cada país
diccionario_maestro = {}

# 3. Agrupamos todos los equipos que existen en la base de datos (tanto locales como visitantes)
todas_las_selecciones = set(df_entrenamiento['home_team'].unique()).union(set(df_entrenamiento['away_team'].unique()))

for seleccion in todas_las_selecciones:
    # Buscamos los partidos donde participó esta selección
    partidos_equipo = df_entrenamiento[(df_entrenamiento['home_team'] == seleccion) | (df_entrenamiento['away_team'] == seleccion)]
    
    if not partidos_equipo.empty:
        # Tomamos el último partido de su historia (el más reciente en la línea de tiempo)
        ultimo_partido = partidos_equipo.sort_values('date').iloc[-1]
        
        # Extraemos sus métricas dependiendo de si en ese último partido jugó como local o visitante
        if ultimo_partido['home_team'] == seleccion:
            goles_totales_marcados_10 = ultimo_partido['goles_totales_marcados_local_10']
            goles_totales_concedidos_10 = ultimo_partido['goles_totales_concedidos_local_10']
            forma_puntos_10 = ultimo_partido['forma_puntos_local_10']
            puntos_fifa = ultimo_partido['puntos_fifa_local']
            ranking_pos = ultimo_partido['ranking_pos_local']
        else:
            goles_totales_marcados_10 = ultimo_partido['goles_totales_marcados_visitante_10']
            goles_totales_concedidos_10 = ultimo_partido['goles_totales_concedidos_visitante_10']
            forma_puntos_10 = ultimo_partido['forma_puntos_visitante_10']
            puntos_fifa = ultimo_partido['puntos_fifa_visitante']
            ranking_pos = ultimo_partido['ranking_pos_visitante']
            
        # Guardamos su "ficha técnica" en el diccionario maestro
        diccionario_maestro[seleccion] = {
            'goles_recientes': goles_totales_marcados_10,
            'goles_concedidos': goles_totales_concedidos_10,
            'forma_reciente': forma_puntos_10,
            'puntos_fifa': puntos_fifa,
            'ranking_fifa': ranking_pos
        }


In [13]:
def predecir_encuentro_mundial(local, visitante):
    # 1. Verificar si ambas selecciones existen en el diccionario maestro
    if local not in diccionario_maestro or visitante not in diccionario_maestro:
        return f"Error: Uno o ambos equipos ('{local}' / '{visitante}') no tienen suficiente historial en el CSV."
        
    # 2. Extraer los datos guardados de cada país de forma automatizada
    loc = diccionario_maestro[local]
    vis = diccionario_maestro[visitante]
    
    # 3. Construir la fila de 11 predictores calculando las diferencias (Locales menos Visitantes)
    datos_partido = {
        'importancia_torneo': 4.0, # Nivel de máxima importancia (Mundial)     
        'goles_totales_marcados_local_10': loc['goles_recientes'],
        'goles_totales_concedidos_local_10': loc['goles_concedidos'],
        'forma_puntos_local_10' : loc['forma_reciente'],
        'puntos_fifa_local' : loc ['puntos_fifa'],
        'ranking_pos_local' : loc ['ranking_fifa'],
        ##visitante
        'goles_totales_marcados_visitante_10': vis['goles_recientes'],
        'goles_totales_concedidos_visitante_10': vis['goles_concedidos'],
        'forma_puntos_visitante_10' : vis['forma_reciente'],
        'puntos_fifa_visitante' : vis ['puntos_fifa'],
        'ranking_pos_visitante' : vis ['ranking_fifa'],
        # Calculamos los predictores diferenciales automáticos
        'dif_goles_marcados_10': loc['goles_recientes'] - vis['goles_recientes'],
        'dif_forma_puntos_10': loc['forma_reciente'] - vis['forma_reciente'],
        'dif_goles_concedidos_10': loc['goles_concedidos'] - vis['goles_concedidos'],
        'dif_puntos_fifa': loc['puntos_fifa'] - vis['puntos_fifa'],
        'dif_ranking_fifa': loc['ranking_fifa'] - vis['ranking_fifa']
    }
    
    # Convertimos en el DataFrame estructurado que exige tu HistGradientBoosting
    X_partido = pd.DataFrame([datos_partido])
    
    # 4. Extraer las probabilidades asignadas por tu modelo entrenado
    probabilidades = modelo_gb.predict_proba(X_partido[features])[0]
    
    p_empate = probabilidades[0]
    p_local = probabilidades[1]
    p_visitante = probabilidades[2]

    print(f"Probabilidad de Victoria de {local}: {p_local*100:.2f}%")
    print(f"Probabilidad de Empate: {p_empate*100:.2f}%")
    print(f"Probabilidad de Victoria de {visitante}: {p_visitante*100:.2f}%\n")
    
    return {'local': p_local, 'empate': p_empate, 'visitante': p_visitante}




In [14]:
import joblib

# Guardamos el modelo entrenado en un archivo físico
joblib.dump(modelo_gb, "modelo_gb.pkl")
# Para cargarlo en el futuro, simplemente usamos:
# modelo_cargado = joblib.load("modelo_gb.pkl")

['modelo_gb.pkl']

In [15]:
df_mundial = pd.read_csv('data/partidos_entrenamiento.csv')
df_mundial = df_mundial[df_mundial['date'] >= '2026-06-01'].copy()
df_mundial

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,puntos_fifa_local,...,goles_totales_marcados_local_10,goles_totales_concedidos_local_10,forma_puntos_local_10,goles_totales_marcados_visitante_10,goles_totales_concedidos_visitante_10,forma_puntos_visitante_10,dif_goles_marcados_10,dif_goles_concedidos_10,dif_forma_puntos_10,importancia_torneo
30511,2026-06-11,South Korea,Czech Republic,NaN,NaN,FIFA World Cup,Zapopan,Mexico,True,631.567156,...,15.0,12.0,19.0,18.0,12.0,16.0,-3.0,0.0,3.0,4
30512,2026-06-11,Mexico,South Africa,NaN,NaN,FIFA World Cup,Mexico City,Mexico,False,1652.330000,...,11.0,10.0,14.0,15.0,11.0,15.0,-4.0,-1.0,-1.0,4
30513,2026-06-12,Canada,Bosnia and Herzegovina,NaN,NaN,FIFA World Cup,Toronto,Canada,False,1461.740000,...,11.0,4.0,17.0,21.0,11.0,16.0,-10.0,-7.0,1.0,4
30514,2026-06-12,United States,Paraguay,NaN,NaN,FIFA World Cup,Inglewood,United States,False,631.567156,...,17.0,16.0,16.0,10.0,10.0,14.0,7.0,6.0,2.0,4
30515,2026-06-13,Australia,Turkey,NaN,NaN,FIFA World Cup,Vancouver,Canada,True,1571.290000,...,15.0,9.0,21.0,21.0,14.0,22.0,-6.0,-5.0,-1.0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30578,2026-06-27,Panama,England,NaN,NaN,FIFA World Cup,East Rutherford,United States,True,1482.100000,...,12.0,7.0,15.0,20.0,2.0,19.0,-8.0,5.0,-4.0,4
30579,2026-06-27,Jordan,Argentina,NaN,NaN,FIFA World Cup,Arlington,United States,True,1374.130000,...,16.0,9.0,17.0,20.0,3.0,19.0,-4.0,6.0,-2.0,4
30580,2026-06-27,DR Congo,Uzbekistan,NaN,NaN,FIFA World Cup,Atlanta,United States,True,631.567156,...,10.0,3.0,17.0,13.0,5.0,15.0,-3.0,-2.0,2.0,4
30581,2026-06-27,Colombia,Portugal,NaN,NaN,FIFA World Cup,Miami Gardens,United States,True,1669.440000,...,20.0,9.0,16.0,22.0,7.0,17.0,-2.0,2.0,-1.0,4


In [ ]:
import pandas as pd
import numpy as np
import joblib

# 1. Tus líneas iniciales para recortar los partidos futuros del Mundial
df_entrenamiento = pd.read_csv("data/partidos_entrenamiento.csv", parse_dates=["date"])
df_mundial = df_entrenamiento[df_entrenamiento["date"] >= "2026-06-01"].copy()

# =====================================================================
# 2. DICCIONARIO OFICIAL DE GRUPOS DEL MUNDIAL 2026
# =====================================================================
# Mapeamos cada selección con su grupo correspondiente (A hasta L)
mapeo_grupos_oficiales = {
    # Grupo A
    "Mexico": "A",
    "South Africa": "A",
    "South Korea": "A",
    "Czech Republic": "A",  # (Ejemplo: ajusta los integrantes reales de cada grupo aquí)
    # Grupo B
    "Canada": "B",
    "Bosnia and Herzegovina": "B",
    "Qatar": "B",
    "Switzerland": "B",
    # Grupo C
    "Brazil": "C",
    "Morocco": "C",
    "Haiti": "C",
    "Scotland": "C",
    # Grupo D
    "United States": "D",
    "Paraguay": "D",
    "Australia": "D",
    "Turkey": "D",
    # Grupo E
    "Germany": "E",
    "Curaçao": "E",
    "Ivory Coast": "E",
    "Ecuador": "E",
    # Grupo F
    "Netherlands": "F",
    "Japan": "F",
    "Sweden": "F",
    "Tunisia": "F",
    # Grupo G
    "Belgium": "G",
    "Egypt": "G",
    "Iran": "G",
    "New Zealand": "G",
    # Grupo H
    "Spain": "H",
    "Cape Verde": "H",
    "Uruguay": "H",
    "Saudi Arabia": "H",
    # Grupo I
    "France": "I",
    "Senegal": "I",
    "Iraq": "I",
    "Norway": "I",
    # Grupo J
    "Argentina": "J",
    "Algeria": "J",
    "Austria": "J",
    "Jordan": "J",
    # Grupo K
    "Portugal": "K",
    "DR Congo": "K",
    "Uzbekistan": "K",
    "Colombia": "K",
    # Grupo L
    "England": "L",
    "Ghana": "L",
    "Croatia": "L",
    "Panama": "L",
    # ... Completa el resto de selecciones según el sorteo oficial de tu proyecto
}

# Inyectamos la columna 'grupo' asociándola al equipo que juegue como local administrativo
df_mundial["grupo"] = df_mundial["home_team"].map(mapeo_grupos_oficiales)

# Por seguridad, si alguna selección no se mapeó, le ponemos 'Desconocido'
df_mundial["grupo"] = df_mundial["grupo"].fillna("Otros")

# =====================================================================
# 3. CONFIGURACIÓN DE FEATURES Y PREDICCIÓN SIMÉTRICA (DATA AUGMENTATION)
# =====================================================================
modelo_gb = joblib.load("modelo_gb.pkl")

features = [
    "goles_totales_marcados_local_10",
    "goles_totales_concedidos_local_10",
    "forma_puntos_local_10",
    "goles_totales_marcados_visitante_10",
    "goles_totales_concedidos_visitante_10",
    "forma_puntos_visitante_10",
    "dif_goles_marcados_10",
    "dif_goles_concedidos_10",
    "dif_forma_puntos_10",
    "puntos_fifa_local",
    "puntos_fifa_visitante",
    "dif_puntos_fifa",
    "ranking_pos_local",
    "ranking_pos_visitante",
    "importancia_torneo",
]

# Creamos la versión invertida para romper el sesgo del orden del dataset
df_mundial_inv = df_mundial.copy()
df_mundial_inv["home_team"] = df_mundial["away_team"]
df_mundial_inv["away_team"] = df_mundial["home_team"]
df_mundial_inv["goles_totales_marcados_local_10"] = df_mundial[
    "goles_totales_marcados_visitante_10"
]
df_mundial_inv["goles_totales_concedidos_local_10"] = df_mundial[
    "goles_totales_concedidos_visitante_10"
]
df_mundial_inv["forma_puntos_local_10"] = df_mundial["forma_puntos_visitante_10"]
df_mundial_inv["goles_totales_marcados_visitante_10"] = df_mundial[
    "goles_totales_marcados_local_10"
]
df_mundial_inv["goles_totales_concedidos_visitante_10"] = df_mundial[
    "goles_totales_concedidos_local_10"
]
df_mundial_inv["forma_puntos_visitante_10"] = df_mundial["forma_puntos_local_10"]
df_mundial_inv["dif_goles_marcados_10"] = df_mundial["dif_goles_marcados_10"] * -1
df_mundial_inv["dif_goles_concedidos_10"] = df_mundial["dif_goles_concedidos_10"] * -1
df_mundial_inv["dif_forma_puntos_10"] = df_mundial["dif_forma_puntos_10"] * -1
df_mundial_inv["puntos_fifa_local"] = df_mundial["puntos_fifa_visitante"]
df_mundial_inv["puntos_fifa_visitante"] = df_mundial["puntos_fifa_local"]
df_mundial_inv["dif_puntos_fifa"] = df_mundial["dif_puntos_fifa"] * -1
df_mundial_inv["ranking_pos_local"] = df_mundial["ranking_pos_visitante"]
df_mundial_inv["ranking_pos_visitante"] = df_mundial["ranking_pos_local"]

# Sacamos las probabilidades cruzadas
probs_a = modelo_gb.predict_proba(df_mundial[features])
probs_b = modelo_gb.predict_proba(df_mundial_inv[features])

# Alineamos y promediamos probabilidades
probs_finales = np.zeros_like(probs_a)
probs_finales[:, 0] = (probs_a[:, 0] + probs_b[:, 0]) / 2  # Empate
probs_finales[:, 1] = (probs_a[:, 1] + probs_b[:, 2]) / 2  # Gana Local original
probs_finales[:, 2] = (probs_a[:, 2] + probs_b[:, 1]) / 2  # Gana Visitante original

df_mundial["p_empate"] = probs_finales[:, 0]
df_mundial["p_local"] = probs_finales[:, 1]
df_mundial["p_visitante"] = probs_finales[:, 2]


# =====================================================================
# 4. SIMULACIÓN DE MONTECARLO CON FILTRO POR GRUPOS
# =====================================================================
def resolver_partido_montecarlo(p_empate, p_local, p_visitante):
    dado = np.random.rand()
    if dado < p_empate:
        return 0
    elif dado < (p_empate + p_local):
        return 1
    else:
        return 2


N_SIMULACIONES = 10000
conteo_clasificados = {}
print(f"🎰 Ejecutando {N_SIMULACIONES} simulaciones analizando las tablas por Grupo...")

for sim in range(N_SIMULACIONES):
    # Puntos desde cero para cada selección participante
    todas_selecciones = pd.concat(
        [df_mundial["home_team"], df_mundial["away_team"]]
    ).unique()
    puntos_sim = {equipo: 0 for equipo in todas_selecciones}

    for idx, partido in df_mundial.iterrows():
        loc = partido["home_team"]
        vis = partido["away_team"]

        res = resolver_partido_montecarlo(
            partido["p_empate"], partido["p_local"], partido["p_visitante"]
        )

        if res == 1:
            puntos_sim[loc] += 3
        elif res == 2:
            puntos_sim[vis] += 3
        else:
            puntos_sim[loc] += 1
            puntos_sim[vis] += 1

    # Armamos la tabla de posiciones de esta iteración concreta
    df_posiciones = pd.DataFrame(list(puntos_sim.items()), columns=["equipo", "puntos"])
    df_posiciones["grupo"] = (
        df_posiciones["equipo"].map(mapeo_grupos_oficiales).fillna("Otros")
    )

    # Ordenamos de forma estricta por grupo y puntos conseguidos
    df_posiciones = df_posiciones.sort_values(
        by=["grupo", "puntos"], ascending=[True, False]
    )

    # Extraemos el Top 2 de cada grupo (los clasificados directos)
    clasificados = df_posiciones.groupby("grupo").head(2)["equipo"].values

    for equipo in clasificados:
        conteo_clasificados[equipo] = conteo_clasificados.get(equipo, 0) + 1

# =====================================================================
# 5. MOSTRAR RESULTADOS ORDENADOS POR GRUPO
# =====================================================================
df_final = pd.DataFrame(
    [
        {
            "Selección": eq,
            "Grupo": mapeo_grupos_oficiales.get(eq, "Otros"),
            "Probabilidad_Clasificar_%": (veces_clasificado / N_SIMULACIONES) * 100,
        }
        for eq, veces_clasificado in conteo_clasificados.items()
    ]
)

df_final = df_final.sort_values(
    by=["Grupo", "Probabilidad_Clasificar_%"], ascending=[True, False]
)

print("\n🏆 PROBABILIDAD DE CLASIFICAR A LA SIGUIENTE RONDA POR GRUPOS:")
print(
    "================================================================================="
)
print(df_final.to_string(index=False))

🎰 Ejecutando 10000 simulaciones analizando las tablas por Grupo...

🏆 PROBABILIDAD DE CLASIFICAR A LA SIGUIENTE RONDA POR GRUPOS:
             Selección Grupo  Probabilidad_Clasificar_%
                Mexico     A                      78.26
        Czech Republic     A                      57.91
          South Africa     A                      38.03
           South Korea     A                      25.80
           Switzerland     B                      75.88
                Canada     B                      54.19
                 Qatar     B                      50.98
Bosnia and Herzegovina     B                      18.95
                Brazil     C                      83.79
               Morocco     C                      61.46
              Scotland     C                      34.43
                 Haiti     C                      20.32
             Australia     D                      78.61
                Turkey     D                      56.83
              Paraguay     D  

In [ ]:
# =====================================================================
# 6. EXTENSIÓN: SIMULACIÓN DE DIECISEISAVOS UTILIZANDO TU MODELO
# =====================================================================

def predecir_cruce_directo(equipo_A, equipo_B):
    """
    Busca los datos más recientes de ambos equipos en el dataframe original,
    construye las variables, aplica simetría y predice quién avanza (Muerte Súbita).
    """
    # Buscamos la última fila donde aparezca cada equipo en el dataset para heredar sus métricas actuales
    fila_A = df_entrenamiento[(df_entrenamiento['home_team'] == equipo_A) | (df_entrenamiento['away_team'] == equipo_A)].iloc[-1]
    fila_B = df_entrenamiento[(df_entrenamiento['home_team'] == equipo_B) | (df_entrenamiento['away_team'] == equipo_B)].iloc[-1]
    
    # Extraemos métricas base de A
    if fila_A['home_team'] == equipo_A:
        g_m_A, g_c_A, f_A, p_f_A, r_A = fila_A['goles_totales_marcados_local_10'], fila_A['goles_totales_concedidos_local_10'], fila_A['forma_puntos_local_10'], fila_A['puntos_fifa_local'], fila_A['ranking_pos_local']
    else:
        g_m_A, g_c_A, f_A, p_f_A, r_A = fila_A['goles_totales_marcados_visitante_10'], fila_A['goles_totales_concedidos_visitante_10'], fila_A['forma_puntos_visitante_10'], fila_A['puntos_fifa_visitante'], fila_A['ranking_pos_visitante']
        
    # Extraemos métricas base de B
    if fila_B['home_team'] == equipo_B:
        g_m_B, g_c_B, f_B, p_f_B, r_B = fila_B['goles_totales_marcados_local_10'], fila_B['goles_totales_concedidos_local_10'], fila_B['forma_puntos_local_10'], fila_B['puntos_fifa_local'], fila_B['ranking_pos_local']
    else:
        g_m_B, g_c_B, f_B, p_f_B, r_B = fila_B['goles_totales_marcados_visitante_10'], fila_B['goles_totales_concedidos_visitante_10'], fila_B['forma_puntos_visitante_10'], fila_B['puntos_fifa_visitante'], fila_B['ranking_pos_visitante']

    # Construimos Escenario Original (A vs B)
    datos_a = {
        'goles_totales_marcados_local_10': g_m_A, 'goles_totales_concedidos_local_10': g_c_A, 'forma_puntos_local_10': f_A, 'puntos_fifa_local': p_f_A, 'ranking_pos_local': r_A,
        'goles_totales_marcados_visitante_10': g_m_B, 'goles_totales_concedidos_visitante_10': g_c_B, 'forma_puntos_visitante_10': f_B, 'puntos_fifa_visitante': p_f_B, 'ranking_pos_visitante': r_B,
        'dif_goles_marcados_10': g_m_A - g_m_B, 'dif_goles_concedidos_10': g_c_A - g_c_B, 'dif_forma_puntos_10': f_A - f_B, 'dif_puntos_fifa': p_f_A - p_f_B, 'dif_ranking_pos': r_A - r_B
    }
    
    # Construimos Escenario Invertido (B vs A)
    datos_b = {
        'goles_totales_marcados_local_10': g_m_B, 'goles_totales_concedidos_local_10': g_c_B, 'forma_puntos_local_10': f_B, 'puntos_fifa_local': p_f_B, 'ranking_pos_local': r_B,
        'goles_totales_marcados_visitante_10': g_m_A, 'goles_totales_concedidos_visitante_10': g_c_A, 'forma_puntos_visitante_10': f_A, 'puntos_fifa_visitante': p_f_A, 'ranking_pos_visitante': r_A,
        'dif_goles_marcados_10': g_m_B - g_m_A, 'dif_goles_concedidos_10': g_c_B - g_c_A, 'dif_forma_puntos_10': f_B - f_A, 'dif_puntos_fifa': p_f_B - p_f_A, 'dif_ranking_pos': r_B - r_A
    }

    # Predicción probabilística dual
    prob_a = modelo_gb.predict_proba(pd.DataFrame([datos_a])[features])[0]
    prob_b = modelo_gb.predict_proba(pd.DataFrame([datos_b])[features])[0]
    
    # Promediamos omitiendo el sesgo de localidad administrativa
    p_ganador_A = (prob_a[1] + prob_b[2]) / 2
    p_ganador_B = (prob_a[2] + prob_b[1]) / 2
    
    # Forzamos Muerte Súbita eliminando el empate comercial
    suma_prob = p_ganador_A + p_ganador_B
    if suma_prob > 0:
        p_A_norm = p_ganador_A / suma_prob
    else:
        p_A_norm = 0.5
        
    # El dado de Montecarlo decide el clasificado final
    if np.random.rand() < p_A_norm:
        return equipo_A
    else:
        return equipo_B

# --- BUCLE PRINCIPAL DE MONTECARLO CON CUADRO REAL DE ELIMINACIÓN ---
conteo_dieciseisavos = {}
conteo_octavos = {}

print(f" Ejecutando {N_SIMULACIONES} simulaciones ")

for sim in range(N_SIMULACIONES):
    # 1. Simular Fase de Grupos con asignación aleatoria de goles para desempates estricto
    puntos_sim = {equipo: 0 for equipo in mapeo_grupos_oficiales.keys()}
    dif_goles_sim = {equipo: 0 for equipo in mapeo_grupos_oficiales.keys()}

    for idx, partido in df_mundial.iterrows():
        loc, vis = partido['home_team'], partido['away_team']
        res = resolver_partido_montecarlo(partido['p_empate'], partido['p_local'], partido['p_visitante'])
        
        if res == 1:
            puntos_sim[loc] += 3
            dg = np.random.choice([1, 2, 3], p=[0.5, 0.3, 0.2])
            dif_goles_sim[loc] += dg; dif_goles_sim[vis] -= dg
        elif res == 2:
            puntos_sim[vis] += 3
            dg = np.random.choice([1, 2, 3], p=[0.5, 0.3, 0.2])
            dif_goles_sim[vis] += dg; dif_goles_sim[loc] -= dg
        else:
            puntos_sim[loc] += 1; puntos_sim[vis] += 1

    # Estructuramos la tabla de posiciones oficial de esta iteración
    df_pos = pd.DataFrame(list(puntos_sim.items()), columns=['equipo', 'puntos'])
    df_pos['dif_goles'] = df_pos['equipo'].map(dif_goles_sim)
    df_pos['grupo'] = df_pos['equipo'].map(mapeo_grupos_oficiales)
    df_pos = df_pos.sort_values(by=['grupo', 'puntos', 'dif_goles'], ascending=[True, False, False])
    
    # Guardamos los líderes absolutos de cada grupo en un diccionario dinámico para armar el cuadro
    # ej: lideres['A'] nos dará la selección que quedó 1ª en el Grupo A en ESTA simulación
    lideres = df_pos.groupby('grupo').nth(0).set_index(df_pos.groupby('grupo').nth(0).index)['equipo'].to_dict()
    segundos = df_pos.groupby('grupo').nth(1).set_index(df_pos.groupby('grupo').nth(1).index)['equipo'].to_dict()
    
    # Extraemos todos los terceros lugares y nos quedamos con los 8 mejores del mundo
    df_terceros = df_pos.groupby('grupo').nth(2).reset_index()
    mejores_terceros = df_terceros.sort_values(by=['puntos', 'dif_goles'], ascending=[False, False]).head(8)['equipo'].tolist()
    
    # --- CONSTRUCCIÓN DE LOS 16 CRUCES EXACTOS (Según el Fixture de la Imagen) ---
    # Asignamos los mejores terceros de forma simplificada rellenando los espacios de la FIFA
    t = mejores_terceros + ["Desconocido"]*8 # Backup por seguridad estructural
    
    cruces_r32 = [
        (lideres.get('A'), t[0]),   # Cruce 1: 1A vs Mejor Tercero
        (segundos.get('B'), segundos.get('F')), # Cruce 2: 2B vs 2F
        (lideres.get('E'), t[1]),   # Cruce 3: 1E vs Mejor Tercero
        (segundos.get('A'), segundos.get('C')), # Cruce 4: 2A vs 2C
        (lideres.get('I'), t[2]),   # Cruce 5: 1I vs Mejor Tercero
        (segundos.get('E'), segundos.get('H')), # Cruce 6: 2E vs 2H
        (lideres.get('F'), segundos.get('C')), # Cruce 7: 1F vs 2C (Ajustado según llaves comunes)
        (lideres.get('K'), segundos.get('L')), # Cruce 8: 1K vs 2L
        (lideres.get('D'), t[3]),   # Cruce 9: 1D vs Mejor Tercero
        (segundos.get('G'), segundos.get('K')), # Cruce 10: 2G vs 2K
        (lideres.get('H'), t[4]),   # Cruce 11: 1H vs Mejor Tercero
        (segundos.get('D'), segundos.get('J')), # Cruce 12: 2D vs 2J
        (lideres.get('G'), t[5]),   # Cruce 13: 1G vs Mejor Tercero
        (lideres.get('L'), segundos.get('I')), # Cruce 14: 1L vs 2I
        (lideres.get('C'), t[6]),   # Cruce 15: 1C vs Mejor Tercero
        (lideres.get('B'), t[7])    # Cruce 16: 1B vs Mejor Tercero
    ]
    
    # Registramos quiénes consiguieron entrar al cuadro de 32
    for equipo_local, equipo_visitante in cruces_r32:
        if equipo_local: conteo_dieciseisavos[equipo_local] = conteo_dieciseisavos.get(equipo_local, 0) + 1
        if equipo_visitante: conteo_dieciseisavos[equipo_visitante] = conteo_dieciseisavos.get(equipo_visitante, 0) + 1
        
        # --- PREDECIR EL GANADOR USANDO TU MODELO AL VUELO ---
        if equipo_local and equipo_visitante and equipo_visitante != "Desconocido":
            ganador_cruce = predecir_cruce_directo(equipo_local, equipo_visitante)
            conteo_octavos[ganador_cruce] = conteo_octavos.get(ganador_cruce, 0) + 1

# =====================================================================
# 7. VISUALIZACIÓN DE PROBABILIDADES REALES DE PLAYOFFS
# =====================================================================
df_playoffs = pd.DataFrame([
    {
        "Selección": eq,
        "Grupo": mapeo_grupos_oficiales.get(eq, "Otros"),
        "Prob_Clasificar_R32_%": (conteo_dieciseisavos.get(eq, 0) / N_SIMULACIONES) * 100,
        "Prob_Alcanzar_Octavos_%": (votos / N_SIMULACIONES) * 100
    }
    for eq, votos in conteo_octavos.items()
])
print(df_playoffs)
#print("\n🏆 PROBABILIDAD DE LLEGAR A OCTAVOS DE FINAL PREDECIDA POR TU MODELO:")
#print("=================================================================================")
#
#print(df_playoffs.sort_values(by="Prob_Alcanzar_Octavos_%", ascending=False).head(20).to_string(index=False))

 Ejecutando 10000 simulaciones 
Empty DataFrame
Columns: []
Index: []
